In [7]:
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path(r"D:\others\safety-vision-api\data\safety.db")
print("Exists:", DB_PATH.exists(), "| Size:", DB_PATH.stat().st_size if DB_PATH.exists() else 0, "bytes")

conn = sqlite3.connect(f"file:{DB_PATH.as_posix()}?mode=ro", uri=True)


def q(sql):
    return pd.read_sql_query(sql, conn)


q("SELECT name FROM sqlite_master WHERE type = 'table'")

Exists: True | Size: 61440 bytes


,name
0,predictions
1,sqlite_sequence
2,detections
3,alerts


In [8]:
# 1. Violations and rate per location (GROUP BY)
q("""
SELECT location,
       COUNT(*) AS images,
       SUM(num_faces) AS faces,
       SUM(num_violations) AS violations,
       ROUND(1.0 * SUM(num_violations) / SUM(num_faces), 3) AS violation_rate
FROM predictions
GROUP BY location
ORDER BY violation_rate DESC
""")

,location,images,faces,violations,violation_rate
0,ward_1,25,205,50,0.244
1,ward_2,15,51,10,0.196
2,icu,20,103,11,0.107


In [9]:
# 2. Detections per class with average confidence (JOIN)
q("""
SELECT d.class,
       COUNT(*) AS detections,
       ROUND(AVG(d.confidence), 3) AS avg_conf
FROM detections d
JOIN predictions p ON p.id = d.prediction_id
GROUP BY d.class
ORDER BY detections DESC
""")

,class,detections,avg_conf
0,with_mask,288,0.735
1,without_mask,59,0.690
2,mask_weared_incorrect,12,0.697


In [10]:
# 3. Alerts with the image that triggered them (JOIN)
q("""
SELECT a.created_at, a.location, a.rule, p.filename, p.num_violations
FROM alerts a
JOIN predictions p ON p.id = a.prediction_id
ORDER BY a.id DESC
""")

,created_at,location,rule,filename,num_violations
0,2026-10-08T09:25:15+00:00,ward_2,high_violation_rate,maksssksksss327.png,6
1,2026-10-08T09:25:15+00:00,ward_2,multiple_violations,maksssksksss327.png,6
2,2026-10-08T09:25:13+00:00,icu,multiple_violations,maksssksksss151.png,3
3,2026-10-08T09:25:12+00:00,ward_1,high_violation_rate,maksssksksss12.png,3
4,2026-10-08T09:25:12+00:00,ward_1,multiple_violations,maksssksksss104.png,6


In [11]:
# 4. Latency percentiles: try writing this one yourself
q("SELECT latency_ms FROM predictions")["latency_ms"].describe(percentiles=[0.5, 0.95, 0.99])

count      60.000000
mean       54.738333
std       173.630264
min        21.700000
50%        30.550000
95%        48.770000
99%       598.280000
max      1375.900000
Name: latency_ms, dtype: float64